In [ ]:
import xarray as xr
import pathlib
import numpy as np
import os
import src.utils
import copy

## specify filepath for data
DATA_FP = pathlib.Path(os.environ["DATA_FP"])

## Load data

In [ ]:
Th_3d = xr.open_mfdataset(
    sorted(list(pathlib.Path(DATA_FP / "cesm"/ "Th_3D").glob("*nc"))),
    combine="nested",
    concat_dim="member",
)

## update coordinates to match
Th_3d = Th_3d.assign_coords({"member":np.arange(100)}).compute()
Th_3d = Th_3d.rename({"h":"thc"})

### Align times

In [ ]:
## Get valid times
valid_time = xr.date_range(
    start="1850-01-01",
    freq="MS",
    periods=3012,
    calendar="noleap",
    use_cftime=True,
)

Th_3d = Th_3d.assign_coords({"time":valid_time})

### compute indices

In [ ]:
def avg(x, lons): 
    """average over longitudes"""
    return x.sel(longitude=slice(*lons)).mean(["longitude"])

def get_Th_indices(x, T_var="T_50"):
    """ compute T,h indices using specified T-variable"""

    indices = xr.merge(
        [
            avg(Th_3d[T_var], (210,270)).rename("T_3"),
            avg(Th_3d[T_var], (190,240)).rename("T_34"),
            avg(Th_3d[T_var], (160,210)).rename("T_4"),
            avg(Th_3d[T_var], (230,280)).rename("T_e"),
            avg(Th_3d[T_var], (180,230)).rename("T_w"),
            avg(Th_3d["thc"], (120,280)).rename("h"),
            avg(Th_3d["thc"], (120,180)).rename("h_w"),
        ]
    )

    return indices

In [ ]:
Th_idx_50 = get_Th_indices(Th_3d, T_var="T_50")
Th_idx_80 = get_Th_indices(Th_3d, T_var="T_80")

## Save to file

In [ ]:
for (x,fname) in [(Th_3d, "Th_3D.nc"), (Th_idx_50, "Th_50.nc"), (Th_idx_80, "Th_80.nc")]:

    fp = pathlib.Path(DATA_FP, "cesm", fname)
    if fp.is_file():
        pass
    else:
        x.to_netcdf(fp)